In [1]:
# =========================
# Week 2: AQI Prediction
# Eco-Aware Personal Assistant
# =========================

# --- Step 0: Import Libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import joblib

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# Load dataset from Week 1 output
df = pd.read_csv("../week1/processed_air_quality.csv")

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

print("✅ Dataset loaded successfully")
df.head()

✅ Dataset loaded successfully


,Date,AQI,Temperature,Humidity,Traffic_Level
0,2025-08-01 00:00:00,152.0,25.0,63,2
1,2025-08-01 01:00:00,229.0,31.0,59,7
2,2025-08-01 02:00:00,142.0,32.0,88,6
3,2025-08-01 03:00:00,64.0,32.0,84,4
4,2025-08-01 04:00:00,156.0,34.0,42,7


In [3]:
# Extract time-based features
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Hour'] = df['Date'].dt.hour

# Drop Date column (not needed for ML model)
df = df.drop(columns=['Date'])

print("✅ Feature engineering complete")
df.head()

✅ Feature engineering complete


,AQI,Temperature,Humidity,Traffic_Level,Month,Day,Hour
0,152.0,25.0,63,2,8,1,0
1,229.0,31.0,59,7,8,1,1
2,142.0,32.0,88,6,8,1,2
3,64.0,32.0,84,4,8,1,3
4,156.0,34.0,42,7,8,1,4


In [4]:
# Fill missing numeric values with column means
df = df.fillna(df.mean(numeric_only=True))

# Verify no missing values remain
print("✅ Missing values handled (after preprocessing)")
print(df.isnull().sum())

✅ Missing values handled (after preprocessing)
AQI              0
Temperature      0
Humidity         0
Traffic_Level    0
Month            0
Day              0
Hour             0
dtype: int64


In [5]:
# Define features (X) and target (y)
X = df.drop(columns=['AQI'])
y = df['AQI']

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✅ Train-Test split complete")
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

# --- Clean NaN in target y ---
y_train = y_train.fillna(y_train.mean())
y_test = y_test.fillna(y_test.mean())
print("✅ Target variable (y) cleaned")

# --- Impute NaN in features X ---
imputer = SimpleImputer(strategy="mean")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print("✅ Missing values imputed successfully")

# --- Debugging check ---
print("NaN in X_train:", np.isnan(X_train).sum())
print("NaN in y_train:", y_train.isnull().sum())


✅ Train-Test split complete
Training data shape: (576, 6)
Testing data shape: (144, 6)
✅ Target variable (y) cleaned
✅ Missing values imputed successfully
NaN in X_train: 0
NaN in y_train: 0


In [6]:
# --- Linear Regression (Baseline) ---
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lr = lin_reg.predict(X_test)

# --- Random Forest Regressor (Better Model) ---
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

print("✅ Models trained successfully")

✅ Models trained successfully


In [7]:
# Function to evaluate models
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"📊 {model_name} Performance")
    print(f"MAE : {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²  : {r2:.2f}")
    print("-"*40)
    return [mae, rmse, r2]

results = {}
results['Linear Regression'] = evaluate_model(y_test, y_pred_lr, "Linear Regression")
results['Random Forest'] = evaluate_model(y_test, y_pred_rf, "Random Forest")

📊 Linear Regression Performance
MAE : 45.11
RMSE: 54.19
R²  : -0.01
----------------------------------------
📊 Random Forest Performance
MAE : 47.79
RMSE: 57.91
R²  : -0.15
----------------------------------------


In [9]:
# Compare actual vs predicted AQI for 50 test samples
plt.figure(figsize=(12,6))
plt.plot(y_test.values[:50], label="Actual AQI", marker='o')
plt.plot(y_pred_rf[:50], label="Predicted AQI (Random Forest)", marker='x')
plt.title("Actual vs Predicted AQI (First 50 Samples)")
plt.xlabel("Sample Index")
plt.ylabel("AQI")
plt.legend()
plt.savefig("actual_vs_predicted.png", dpi=300, bbox_inches='tight')
plt.close()

print("✅ Plot saved as actual_vs_predicted.png")


✅ Plot saved as actual_vs_predicted.png


In [12]:
# --- Step 8: Save the Trained Model ---
import joblib

# Save the best performing model (Random Forest here)
joblib.dump(rf_reg, "aqi_random_forest_model.pkl")
print("✅ Model saved successfully as 'aqi_random_forest_model.pkl'")

✅ Model saved successfully as 'aqi_random_forest_model.pkl'


In [14]:
# --- Step 9: Save Evaluation Report ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Recalculate metrics (in case Step 6 was not run)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf  = r2_score(y_test, y_pred_rf)

# Save metrics to text file
with open("model_evaluation_report.txt", "w") as f:
    f.write("Model Evaluation Report - Week 2\n")
    f.write("="*40 + "\n")
    f.write(f"Linear Regression:\n MAE={mae_lr:.2f}, MSE={mse_lr:.2f}, R²={r2_lr:.2f}\n\n")
    f.write(f"Random Forest:\n MAE={mae_rf:.2f}, MSE={mse_rf:.2f}, R²={r2_rf:.2f}\n\n")

print("✅ Evaluation report saved as 'model_evaluation_report.txt'")

✅ Evaluation report saved as 'model_evaluation_report.txt'
